# Run a real model on Colab, not on your laptop

For anyone whose machine has 8 GB of RAM. Colab gives you ~12 GB and, on the free
tier, usually a T4 GPU — so the model runs **there** and your laptop only sends
text.

**Do this first, before anything else:**

> **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

Without the GPU this still works, but a 7B model on CPU answers in minutes rather
than seconds, and you will not enjoy the session.

## Why a 7B and not something smaller

Measured on this course's own session 3, same question, same corpus:

| Model | Size | Session 3 |
|---|---|---|
| `llama3.2:1b` | 1.3 GB | **fails** — cannot produce a valid `confidence`, twice, then refuses |
| `qwen2.5:7b-instruct` | 4.7 GB | passes, cites the right document |

The 1B is not broken. It cannot hold a four-field contract, and the parser
correctly refuses what it returns. That is session 3's lesson arriving early —
but it means a 1B is no use for the lab.

## 1. Install Ollama

About 30 seconds. This is the official Linux installer.

In [ ]:
# manual-run: installs Ollama and pulls a 4.7 GB model; it needs a Colab runtime
# and several minutes, so CI reads this notebook rather than executing it.
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start the server in the background

Colab runs one cell at a time, so `ollama serve` has to be detached or it would
block the notebook forever. `nohup` puts it in the background and sends its
output to a log you can read if something goes wrong.

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(
    ["nohup", "ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait for it to answer rather than guessing with a fixed sleep.
for attempt in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print(f"ollama is up after {attempt + 1}s")
        break
    except Exception:
        time.sleep(1)
else:
    print("ollama did not start — read /content/ollama.log")

## 3. Pull the model

4.7 GB, so two to four minutes depending on the runtime. **This happens again
every time Colab gives you a new machine** — the disk does not survive a
disconnect. That is the real cost of this approach, and it is the price of not
buying more RAM.

In [ ]:
!ollama pull qwen2.5:7b-instruct

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU — expect minutes per answer, not seconds"

## 4. Get the course

A shallow clone of the public cohort repository.

In [ ]:
import os
from pathlib import Path

if not Path("/content/dev3pack-cohort-2026-09").exists():
    !git clone --depth 1 https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git /content/dev3pack-cohort-2026-09

os.chdir("/content/dev3pack-cohort-2026-09")
!pip -q install python-dotenv
print("course at", Path.cwd())

## 5. Point the course at your Colab model

`OLLAMA_BASE_URL` stays `localhost` — the server is on the *same machine* as this
notebook, which is the whole trick. Nothing is exposed to the internet and there
is no key anywhere.

In [ ]:
import sys

os.environ["BOOTCAMP_PROVIDER"] = "ollama"
os.environ["BOOTCAMP_MODEL"] = "qwen2.5:7b-instruct"
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434/v1"

sys.path.insert(0, str(Path.cwd() / "src"))

from bootcamp_agent.ollama import OllamaClient, OllamaError

client = OllamaClient(model="qwen2.5:7b-instruct")
print(client.complete(system="Answer in one short sentence.",
                      user="What is a structured output, and why would a program want one?"))

## 6. Prove it on the real thing

The session 3 agent, on the real corpus, through your Colab model. Read the trace:
retrieval first, then the call, then the decision.

In [ ]:
from bootcamp_agent.agent import answer_question
from bootcamp_agent.documents import load_corpus

documents = load_corpus(Path.cwd() / "data" / "corpus")
result = answer_question(
    "What stopping conditions should an agent loop have?", documents, client
)

for event in result.trace:
    print(f"trace[{event.kind}] {event.detail}")

print()
print(result.answer.answer)
print(f"citations={list(result.answer.citations)} "
      f"confidence={result.answer.confidence} "
      f"review={result.answer.needs_human_review}")

### If that came back as a refusal

It is not necessarily broken. A small model that cannot produce the four fields
gets refused by the parser, on purpose — the trace line tells you which field it
got wrong. That **is** session 3.

If you want it to succeed, the model has to be big enough to hold the contract.
`qwen2.5:7b-instruct` does; a 1B does not.

## What this does and does not buy you

**Does:** a real model, on a real GPU, with no key, no card, and nothing
installed on your laptop.

**Does not:** persistence. Colab hands you a fresh machine after a disconnect or
an idle timeout, and you run cells 1–5 again — about three minutes.

**Your homework and submissions still happen on your own machine**, because
`bootcamp submit` needs your git checkout. Use Colab for the parts that need a
model, and your laptop for everything else. Every graded exercise in this course
runs offline on `FakeLLM` and needs no model at all.